In [1]:
import pandas as pd
from ipynb.fs.defs.functions import validate_df

In [2]:
# Tabellen einlesen und mergen
df_hilfen = pd.read_csv("../data/processed/hilfen_2024.csv", sep=",", encoding="UTF-8")
df_bevoelkerung = pd.read_csv("../data/processed/bevoelkerung_2024.csv", sep=",", encoding="UTF-8")
df_sgbii = pd.read_csv("../data/processed/sgb2_2024.csv", sep=",", encoding="UTF-8")
df_arztdichte = pd.read_csv("../data/processed/arztdichte_2024.csv", sep=",", encoding="UTF-8")
df_traeger = pd.read_csv("../data/processed/traeger.csv", sep=",", encoding="UTF-8")
df_bildung = pd.read_csv("../data/processed/bildungsabschluss_2024.csv", sep=",", encoding="UTF-8")
df_dichte = pd.read_csv("../data/processed/bevoelkerungsdichte_2024.csv", sep=",", encoding="UTF-8")

dfs_to_merge = [df_bevoelkerung,df_sgbii, df_arztdichte, df_traeger, df_bildung, df_dichte]

df_merged = df_hilfen.copy()
for df_other in dfs_to_merge:
    df_merged = df_merged.merge(df_other, on="Name", how="left")

print(df_merged.columns)

missing_in_other = set(df_merged["Name"]) - set(df_other["Name"])
extra_in_other   = set(df_other["Name"]) - set(df_merged["Name"])

print("Fehlen in df_other:", list(sorted(missing_in_other))[:20])
print("Nur in df_other:", list(sorted(extra_in_other))[:20])


Index(['Name', 'Typ 1', 'Typ 2', 'Insgesamt', 'Anzahl 35a Hilfen',
       'Gesamtbevölkerung', 'Bevölkerung 6 bis 20', 'Anteil gesamt',
       'SGB II-Bezug', 'Kinderarztdichte', 'Unnamed: 0', 'ROR', 'KJP-Dichte',
       'Überörtlicher Träger', 'Bildungsindex', 'Bevölkerungsdichte'],
      dtype='object')
Fehlen in df_other: []
Nur in df_other: []


In [3]:
# relative Werte berechnen
df_merged["35a Hilfen pro 10000"] = (df_merged["Anzahl 35a Hilfen"] / df_merged["Bevölkerung 6 bis 20"] * 10000).round(0).astype(int)
df_merged["erz. Hilfen pro 10000"] = (df_merged["Insgesamt"] / df_merged["Bevölkerung 6 bis 20"] * 10000).round(0).astype(int)
df_merged["SGB II-Quote"] = (df_merged["SGB II-Bezug"] / df_merged["Gesamtbevölkerung"]*100).round(2)
df_merged["Anteil gesamt"] = df_merged["Anteil gesamt"].round(1)

In [4]:
# Essen imputieren
RUHR = ["Bochum", "Dortmund", "Duisburg", "Mülheim an der Ruhr", "Oberhausen", "Gelsenkirchen"]
subset = df_merged.loc[df_merged["Name"].isin(RUHR), "35a Hilfen pro 10000"]

mean = subset.apply(pd.to_numeric, errors="coerce").mean()

df_merged.loc[df_merged["Name"] == "Essen", "35a Hilfen pro 10000"] = round(mean, 0)

In [5]:
# Spaltennamen anpassen
df_merged=df_merged.rename(columns={"Anteil gesamt": "Anteil Kinder a.d. Gesamtbev.",
                     "Insgesamt": "erz. Hilfen absolut",
                     "Anzahl 35a Hilfen": "35a Hilfen absolut"})
df_merged = df_merged[["Name",
                        "Typ 1",
                        "Typ 2",
                        "ROR",
                        "erz. Hilfen pro 10000",
                        "35a Hilfen pro 10000",
                        "Überörtlicher Träger",
                        "Bevölkerung 6 bis 20",
                        "Anteil Kinder a.d. Gesamtbev.",
                        "SGB II-Quote",
                        "Kinderarztdichte",
                        "KJP-Dichte",
                        "Bildungsindex",
                        "Bevölkerungsdichte"]]

In [6]:
print(df_merged.info())

typ_order = ["Kreisfreie Stadt", "Großer Kreis", "Städteregion", "Kreis"]
df_merged["Typ"] = pd.Categorical(df_merged["Typ 1"], categories=typ_order, ordered=True)
df_merged = df_merged.sort_values(by=["Typ 1", "Name"], ascending=[True, True])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53 entries, 0 to 52
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Name                           53 non-null     object 
 1   Typ 1                          53 non-null     object 
 2   Typ 2                          53 non-null     object 
 3   ROR                            53 non-null     object 
 4   erz. Hilfen pro 10000          53 non-null     int64  
 5   35a Hilfen pro 10000           53 non-null     int64  
 6   Überörtlicher Träger           53 non-null     object 
 7   Bevölkerung 6 bis 20           53 non-null     int64  
 8   Anteil Kinder a.d. Gesamtbev.  53 non-null     float64
 9   SGB II-Quote                   53 non-null     float64
 10  Kinderarztdichte               53 non-null     float64
 11  KJP-Dichte                     53 non-null     float64
 12  Bildungsindex                  53 non-null     float

In [7]:
numerical_cols = ['erz. Hilfen pro 10000','35a Hilfen pro 10000','Bevölkerung 6 bis 20', 'Anteil Kinder a.d. Gesamtbev.','SGB II-Quote', 'Kinderarztdichte', 'Bildungsindex', 'Bevölkerungsdichte']

validate_df(df_merged,
            not_null = numerical_cols,
            positive= numerical_cols,
            numeric= numerical_cols,
            bounds={'Anteil Kinder a.d. Gesamtbev.':(0,100),'SGB II-Quote':(0,100), 'Kinderarztdichte':(0,100), "KJP-Dichte":(0,100), "Bildungsindex":(0,5)},
            key_cols=["Name"],
            df_name="Masterframe")

Masterframe: alle Checks bestanden


[]

In [8]:
df_merged.to_csv("../data/processed/master_2024.csv", index=False)